[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/modal_xpbd/modal_xpbd.ipynb)

# Flexible Bodies, Rigidly Joined

This example computes the motion of articulated structures whose links bend slightly while swinging through large rotations or buckling under axial compression. Each link is modelled as a cross-braced girder carried rigidly by a pose motor while deforming along its lowest vibration modes, with point constraints holding anchor vertices together across adjacent girders. Modal extended position-based dynamics treats every retained vibration mode as a compliant spring constraint eliminated analytically within each step, so stiff modes, soft modes, and exact rigid links share one solver over the joint forces alone.

Rigid motion and small elastic deformation compose through `motor >> local_points`, while advancing a pose motor by a body-frame twist takes the exponential step `motor * (twist_step * -0.5).exp()` ([Gunn, 2011](https://arxiv.org/pdf/1101.4542); [Dorst and De Keninck, 2022](https://bivector.net/PGAdyn.pdf)). At each joint anchor, `motor >> local_anchors.commutator(Twist)` maps an extrinsic body twist directly to an intrinsic world-frame anchor displacement. Because virtual work pairs an extrinsic force plane with an intrinsic displacement as `force & displacement`, the adjugate `rigid.adjugate()` pulls joint reactions straight back to intrinsic body forques without a coordinate transpose or a metric on twists, and `Force & anchor_modes` projects the same reactions onto scalar modal loads.

In the matrix Newton–Euler notation of [Macklin et al., 2016](https://matthias-research.github.io/pages/publications/XPBD.pdf) and [Müller et al., 2020](https://matthias-research.github.io/pages/publications/PBDBodies.pdf), each planar body splits into centre-of-mass position $\mathbf{x}_i$, rotation matrix $\mathbf{R}(\theta_i)$, and modal amplitudes $\mathbf{q}_i$, with joint gap $\mathbf{C}_{ij} = \mathbf{x}_j + \mathbf{R}(\theta_j)(\mathbf{r}_j + \boldsymbol{\Phi}_j \mathbf{q}_j) - \mathbf{x}_i - \mathbf{R}(\theta_i)(\mathbf{r}_i + \boldsymbol{\Phi}_i \mathbf{q}_i)$ differentiated block by block into translational identity blocks, rotated lever-arm blocks $\mathbf{R}(\theta_i)\mathbf{J}_\perp(\mathbf{r}_i + \boldsymbol{\Phi}_i \mathbf{q}_i)$ with $\mathbf{J}_\perp = \bigl[\begin{smallmatrix}0 & -1\\ 1 & 0\end{smallmatrix}\bigr]$, and modal blocks $\mathbf{R}(\theta_i)\boldsymbol{\Phi}_i$. Eliminating the modal compliance $\tilde{\boldsymbol{\alpha}}_m$ from the Karush–Kuhn–Tucker system leaves $(\mathbf{J}_b \mathbf{M}_b^{-1} \mathbf{J}_b^\top + \mathbf{J}_m \mathbf{W}_m \mathbf{J}_m^\top + \tilde{\boldsymbol{\alpha}}_p)\,\boldsymbol{\lambda} = -\mathbf{C} - \mathbf{J}_m \Delta\mathbf{q}^{\mathrm{free}}$ with $\mathbf{W}_m = \tilde{\boldsymbol{\alpha}}_m (\mathbf{I} + \tilde{\boldsymbol{\alpha}}_m)^{-1}$, after which $\mathbf{J}_b^\top \boldsymbol{\lambda}$ updates translations and angles through separate coordinate equations.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from dataclasses import replace

import numpy as np

from numga import NumpyContext, dim, stack
from numga.algebras import PGA2D as ga
from numga.sparse import SparseExtensor, spdiag
from numga.topology import Topology
from examples.mechanics import lie_integrators as lie
from examples.mechanics.modal_xpbd import render
from examples.mechanics.modal_xpbd.scenarios import Bodies, Constraints, Shape, clamped_beam, hinged_chain, truss
from examples.util.animation import save_figure, show_states

mv = NumpyContext(ga, units=True).multivector
Scalar = ga.gatype.scalar()                                                # Scalar+
Point = ga.gatype.antivector()                                             # Antivector-
Twist = ga.gatype.bivector()                                               # Bivector+
Forque = ga.gatype.antibivector()                                          # Antibivector-
Motor = ga.gatype.rotor()                                                  # Rotor+
# A displacement is an ideal point through the null plane w, while a force is a plane through the
# origin; pairing `force & displacement` evaluates the mechanical work along that displacement.
Displacement = ga.gatype.antivector().derive.degenerate                    # Antivector-
Force = ga.gatype.vector().derive.nondegenerate                            # Vector+
frame_duration_ms = 40

## Girder vibration modes

Each girder is modelled as a cross-braced lattice of axial bars with lumped masses at its chord vertices. Solving the generalized eigenproblem of bar stiffness against vertex mass separates planar rigid motions at zero frequency from elastic vibration modes, and dropping those three rigid motions leaves the lowest bending, shearing, and axial modes to deform the girder in its body frame while its pose motor carries rigid translation and rotation.

In [ ]:
# Chain girder geometry, bar stiffness, density, and retained vibration modes.
force_unit = dim.mass * dim.length / dim.time**2
density_unit = dim.mass / dim.length**2
n_bays = 4
length, height = mv.scalar(1.0, unit=dim.length), mv.scalar(0.2, unit=dim.length)
stiffness = mv.scalar(300.0, unit=force_unit)
density = mv.scalar(1.0, unit=density_unit)
n_modes = 8
n_shown_modes, mode_scale = 4, mv.scalar(0.03, unit=dim.length * dim.mass.root(2))

In [ ]:
def girder(n_bays: int, length: Scalar, height: Scalar, stiffness: Scalar, density: Scalar, n_modes: int) -> Shape:
    """Bar stiffness solved against lumped vertex masses reduces a girder of `n_bays` bays to its
    `n_modes` lowest non-rigid vibration modes."""
    positions, edges = truss(mv, n_bays, length, height)                   # Antivector-[V], [E, 2]
    topology = Topology(positions.batch().shape[-1], edges)
    V, E = topology.n_elements
    # Subtracting each bar's tail from its head cancels unit point weights into an ideal displacement.
    ends = topology.d[0]                                                   # Scalar+[E <- V]
    bar_displacements = (ends * positions).cast(Displacement)              # Antivector-[E], length
    # Dualizing once with the Euclidean metric turns each bar displacement into the force plane along
    # that bar, whose pairing with the displacement gives the squared bar length.
    pulls = bar_displacements.dual_inverse()                               # Vector+[E], length
    lengths = (pulls & bar_displacements).square_root()                    # Scalar+[E], length
    unit_forces = pulls / lengths                                          # Vector+[E]
    # Each bar measures axial stretch by pairing its unit force with relative end displacement and
    # pulls back along the same line of action.
    bars = unit_forces * (unit_forces & Displacement) * stiffness / lengths  # (Vector+ <- Antivector-)[E], mass/time^2
    stretching = ends * Displacement                                       # (Antivector- <- Antivector-)[E <- V]
    masses = (density * length * height / V).broadcast_to((V,)).field()    # Scalar+[V], mass
    weights = spdiag(masses * Displacement.dual_inverse())                 # (Vector+ <- Antivector-)[V <- V], mass
    # Pull bar stiffnesses back to the vertices through the coboundary adjugate.
    girder_stiffness = stretching.adjugate()(bars(stretching))             # (Vector+ <- Antivector-)[V <- V], mass/time^2
    # Drop the three zero-frequency planar rigid motions carried by the body's motor.
    squared_frequencies, modes = girder_stiffness.eigh(weights, 3 + n_modes)  # [3 + n_modes] Scalar+, 1/time^2; Antivector-[V], 1/mass^(1/2)
    squared_frequencies, modes = squared_frequencies[3:], modes[3:]        # [n_modes] Scalar+, Antivector-[V]
    centre = (positions * masses).sites.sum() / masses.sites.sum()         # [] Antivector-, 1 | length
    rest = positions - centre + mv.w.dual()                                # Antivector-[V], 1 | length
    inertia = (masses * rest).regressive(rest.commutator(Twist)).sites.sum()  # [] Antibivector- <- Bivector+
    compliance = 1 / squared_frequencies                                   # [n_modes] Scalar+, time^2
    return Shape(rest=rest, modes=modes, compliance=compliance, masses=masses, edges=edges, inertia=inertia)


shape = girder(n_bays, length, height, stiffness, density, n_modes)
save_figure(render.mode_shapes(shape.rest, shape.rest + shape.modes[:n_shown_modes] * mode_scale, shape.edges), "modal_xpbd_modes");

Each panel overlays one displaced eigenmode in colour onto the grey rest girder, progressing from fundamental flexure and S-bending to higher-frequency shear and axial stretching across the cross-braced bays.

## Bodies and point constraints

Each girder body carries a rigid pose motor together with scalar amplitudes for its retained vibration modes, and `motor >> local_points` places the deformed rest vertices in the world frame. A point constraint requires an anchor vertex on one girder to coincide with an anchor vertex on its neighbour, so a single upper-chord point forms a pin hinge while a pair of upper- and lower-chord points forms a moment-resisting splice. Along the hanging chain below, modal compliance starts at zero on an exactly rigid link at the root pin and increases fourfold from each moving link to the next toward the tip.

In [ ]:
# Moving links in the chain, modal damping ratio, per-link flexibility scale, and gravity.
n_links = 4
damping = np.array([0.02])                                                 # [1]
flexibility = np.concatenate([[0.0], 4.0 ** np.arange(n_links - 1)])[None]  # [1, n_links]
gravity = mv.direction([0, -4], unit=dim.length / dim.time**2)             # [] Antivector-, length/time^2

In [ ]:
def world_points(bodies: Bodies, shape: Shape) -> Point:
    """Modal superposition in each body frame followed by the body's motor places all girder
    vertices in the world frame (`[..., n_bodies] Antivector-[V]`)."""
    local_points = shape.rest + (shape.modes[:, None] * bodies.amplitudes.batch()).sum(axis=-2)  # [..., n_bodies] Antivector-[V]
    return bodies.motor.batch() >> local_points                            # [..., n_bodies] Antivector-[V]


bodies, constraints = hinged_chain(mv, shape, damping, flexibility)
initial_points = world_points(bodies, shape)
save_figure(render.chain_figure(initial_points, shape.edges), "modal_xpbd_chain");

The four moving girders extend horizontally from the dark root pivot on the upper chord, coloured blue, teal, amber, and purple from the rigid link at the pin out to the softest link at the free tip.

## Anchor kinematics and joint gaps

Closing the joints requires the sensitivity of each world-frame anchor gap to both rigid body twists and modal amplitudes across the two incident girders. Commuting each deformed body-frame anchor with `Twist` and sandwiching by `motor` maps body twists to world-frame anchor displacements, while sandwiching the anchor mode vectors gives the modal gap sensitivities and `Force & anchor_modes` pulls constraint forces back to scalar modal loads through the incidence adjugate.

In [ ]:
def coupling(bodies: Bodies, constraints: Constraints) -> tuple[SparseExtensor, SparseExtensor, SparseExtensor, Displacement]:
    """Anchor kinematics and constraint incidence assemble the sparse rigid and modal gap maps, the
    modal load pullback, and the current world-frame gaps."""
    motor = constraints.ends * bodies.motor[..., None]                     # [..., 2] Rotor+[n_constraints]
    local_anchors = constraints.anchors + (constraints.modes * (constraints.ends * bodies.amplitudes[..., None])).sum(axis=-2)  # [..., 2] Antivector-[n_constraints]
    signs = np.array([-1, 1])                                              # [2]
    # Commuting each local anchor with `Twist` maps body-frame twists to intrinsic anchor
    # displacements, which `motor` sandwiches into the world frame.
    anchor_motion = (motor >> local_anchors.commutator(Twist)) * signs     # [..., 2] (Antivector- <- Bivector+)[n_constraints]
    anchor_modes = (motor[..., None, :] >> constraints.modes) * signs      # [..., n_modes, 2] Antivector-[n_constraints]
    gap = ((motor >> local_anchors) * signs).sum(axis=-1)                  # [...] Antivector-[n_constraints]
    rigid = (spdiag(anchor_motion) * constraints.ends).sum(axis=-1)        # [...] (Antivector- <- Bivector+)[n_constraints <- n_bodies]
    modal = (spdiag(anchor_modes) * constraints.ends).sum(axis=-1)         # [..., n_modes] Antivector-[n_constraints <- n_bodies]
    # Pairing extrinsic constraint forces with intrinsic modal anchor displacements measures virtual
    # work per unit modal amplitude and pulls it back across `ends.adjugate()`.
    modal_load = (constraints.ends.adjugate() * spdiag(Force & anchor_modes)).sum(axis=-1)  # [..., n_modes] (Scalar+ <- Vector+)[n_bodies <- n_constraints]
    return rigid, modal, modal_load, gap


rigid, modal, modal_load, initial_gap = coupling(bodies, constraints)
print(rigid.gatype, rigid.shape)
print(modal_load.gatype, modal_load.shape)

## Modal compliance and joint projection

Over a step `dt`, each retained mode acts as a unit-mass oscillator whose spring compliance and Rayleigh damping combine into an effective step compliance, predictor residual, and unit-load response. Eliminating the modal amplitudes in closed form leaves one symmetric Schur-complement system over the constraint reactions alone, combining rigid body mobility `rigid(bodies.inverse_inertia(rigid.adjugate()))`, modal mobility across all retained modes, and joint compliance so zero-compliance rigid bodies and stiff splices stay regular.

In [ ]:
def modal_terms(bodies: Bodies, previous_amplitudes: Scalar, dt: float) -> tuple[Scalar, Scalar, Scalar]:
    """Implicit spring and dashpot weights over step `dt` determine each mode's effective
    compliance, predictor residual, and unit-load response (`[..., n_modes] Scalar+[n_bodies]`)."""
    # Unit modal mass gives omega * alpha == sqrt(alpha), vanishing cleanly on rigid bodies (alpha == 0).
    damping = 2 * bodies.damping * bodies.compliance.square_root() / dt    # [..., n_modes] Scalar+[n_bodies]
    compliance = bodies.compliance / dt**2 / (1 + damping)                 # [..., n_modes] Scalar+[n_bodies]
    residual = (bodies.amplitudes + damping * (bodies.amplitudes - previous_amplitudes)) / (1 + damping)  # [..., n_modes] Scalar+[n_bodies]
    response = 1 / (1 + compliance)                                        # [..., n_modes] Scalar+[n_bodies]
    return compliance, residual, response


def project_constraints(bodies: Bodies, constraints: Constraints, previous_amplitudes: Scalar, mode_reactions: Scalar, dt: float) -> tuple[Bodies, Scalar]:
    """Solving the joint Schur complement closes all constraint gaps at once while updating body
    motors, modal amplitudes, and accumulated modal reactions."""
    compliance, residual, response = modal_terms(bodies, previous_amplitudes, dt)  # [..., n_modes] Scalar+[n_bodies], Scalar+[n_bodies], Scalar+[n_bodies]
    rigid, modal, modal_load, gap = coupling(bodies, constraints)
    spring_load = -(residual + compliance * mode_reactions)                # [..., n_modes] Scalar+[n_bodies]
    unconstrained_step = response * spring_load                            # [..., n_modes] Scalar+[n_bodies]
    mobility = spdiag(response * compliance)                               # [..., n_modes] Scalar+[n_bodies <- n_bodies]
    constraint_compliance = constraints.compliance / dt**2 * Force.dual()  # (Antivector- <- Vector+)[n_constraints]
    # Because `rigid.adjugate()(force) & twist == force & rigid(twist)`, the adjugate maps joint
    # forces directly to body forques without a coordinate transpose or metric on twists.
    system = rigid(bodies.inverse_inertia(rigid.adjugate())) + (modal * (mobility * modal_load)).sum(axis=-1) + constraint_compliance  # [...] (Antivector- <- Vector+)[n_constraints <- n_constraints]
    reactions = system.solve((-gap - (modal * unconstrained_step).sum(axis=-1)).cast(Displacement))  # [...] Vector+[n_constraints]
    twist_step = bodies.inverse_inertia(rigid.adjugate()(reactions))       # [...] Bivector+[n_bodies]
    mode_loads = modal_load(reactions[..., None])                          # [..., n_modes] Scalar+[n_bodies]
    reaction_change = response * (spring_load - mode_loads)                # [..., n_modes] Scalar+[n_bodies]
    stepped_motor = bodies.motor * (twist_step * -0.5).exp()               # [...] Rotor+[n_bodies]
    constrained_bodies = replace(
        bodies,
        motor=stepped_motor,
        amplitudes=bodies.amplitudes + mode_loads + reaction_change,
    )
    mode_reactions = mode_reactions + reaction_change                      # [..., n_modes] Scalar+[n_bodies]
    return constrained_bodies, mode_reactions

## Predictor-corrector time step

Each time step integrates free rigid-body motion under gravity on the motor group, advances modal amplitudes by their current rates, relaxes the modal springs, projects all joint and modal constraints together, and recovers updated body twists and modal rates from the finite pose and amplitude changes across the step.

In [ ]:
def step(bodies: Bodies, constraints: Constraints, dt: float, gravity: Displacement) -> Bodies:
    """Free rigid-body prediction under `gravity` followed by simultaneous modal and joint
    projection advances `bodies` by one step `dt`."""
    previous = bodies

    def weight(motor: Motor, rate: Twist) -> Forque:
        # Gravity acts at each body's centre of mass, placed at the origin of the body frame.
        return (mv.w.dual() & (motor << gravity)) * bodies.masses          # [...] Antibivector-[n_bodies]

    motor, _ = lie.explicit_rk4(bodies.motor, bodies.rate, bodies.inertia, bodies.inverse_inertia, dt, weight)
    bodies = replace(bodies, motor=motor, amplitudes=bodies.amplitudes + bodies.rates * dt)
    _, residual, response = modal_terms(bodies, previous.amplitudes, dt)   # [..., n_modes] Scalar+[n_bodies], Scalar+[n_bodies], Scalar+[n_bodies]
    mode_reactions = -residual * response                                  # [..., n_modes] Scalar+[n_bodies]
    bodies, _ = project_constraints(
        replace(bodies, amplitudes=bodies.amplitudes + mode_reactions),
        constraints,
        previous.amplitudes,
        mode_reactions,
        dt,
    )
    return replace(
        bodies,
        rate=(~previous.motor * bodies.motor).log() * (-2 / dt),
        rates=(bodies.amplitudes - previous.amplitudes) / dt,
    )

## Swinging chain

Released from rest in a horizontal pose under downward gravity, the hinged chain swings through large angles carried by the pose motors while bending locally along each link's vibration modes. Rest-frame inertia is held fixed and gyroscopic coupling between rotation and vibration is omitted, an approximation that holds while rigid angular rates stay slow relative to the retained vibration frequencies.

In [ ]:
# Time step, substeps per frame, and frame count for the swinging chain.
dt, n_substeps, n_frames = mv.scalar(0.02, unit=dim.time), 2, 150

In [ ]:
def swing_states(shape: Shape, bodies: Bodies, constraints: Constraints, gravity: Displacement, dt: float, n_substeps: int, n_frames: int) -> Iterator[tuple[Point, Bodies]]:
    """Yields `(world_points, bodies)` across `n_frames` frames under `gravity`, advancing
    `n_substeps` steps per frame."""
    for _ in range(n_frames):
        yield world_points(bodies, shape), bodies
        for _ in range(n_substeps):
            bodies = step(bodies, constraints, dt, gravity)


swing_history = show_states(
    swing_states(shape, bodies, constraints, gravity, dt, n_substeps, n_frames),
    render.swinging_chain,
    shape.edges,
    initial_points,
    name="modal_xpbd_swing",
    duration_ms=frame_duration_ms,
    n_frames=n_frames,
)

As the chain drops and whips through the bottom of its arc, the blue root link remains straight at zero compliance while flexural waves grow progressively across the teal, amber, and purple links toward the free tip without opening the pin hinges.

## Buckling column

Splicing stiff girders end to end between two clamped supports forms a slender column compressed by translating the right-hand support inward a little further on every frame. Whereas a single linear modal body superposes modes in one fixed frame and cannot buckle, a chain of spliced girders carries finite rotations on its pose motors and buckles laterally once axial compression exceeds the Euler critical load. Starting from a straight column perturbed only by floating-point rounding, lateral deflection stays negligible just past the critical threshold while the unstable straight equilibrium amplifies rounding noise into a buckled arch.

In [ ]:
# Moving girders, bays of unit length, chord height, bar stiffness, damping, and compression schedule.
n_beam_girders, n_beam_bays = 8, 32
beam_length = mv.scalar(float(n_beam_bays), unit=dim.length)
beam_height = mv.scalar(1.0, unit=dim.length)
beam_stiffness, beam_damping = mv.scalar(1e11, unit=force_unit), np.array([3.0])
end_displacement = mv.scalar(0.15, unit=dim.length)
n_beam_frames, n_beam_substeps, beam_dt = 90, 4, mv.scalar(0.02, unit=dim.time)

In [ ]:
beam_shape = girder(n_beam_bays, beam_length, beam_height, beam_stiffness, density, n_modes)
beam_bodies, beam_splices = clamped_beam(mv, beam_shape, n_beam_girders, beam_damping)
displacements = end_displacement * np.arange(n_beam_frames) / n_beam_frames  # [n_beam_frames] Scalar+, length


def end_loaded_bodies(bodies: Bodies, rest_motor: Motor, displacement: Scalar) -> Bodies:
    """Translating the right-hand fixed support inward by `displacement` along `x` updates the
    end-loaded column state."""
    motor_step = (mv.xw * (-displacement / 2)).exp()                       # [] Rotor+, 1 | length
    driven_support = rest_motor.batch()[..., -1] * motor_step              # [...] Rotor+
    return replace(bodies, motor=bodies.motor.batch().at[..., -1].set(driven_support).field().normalized())


def compression_states(shape: Shape, bodies: Bodies, constraints: Constraints, displacements: Scalar, dt: Scalar, n_substeps: int) -> Iterator[tuple[Point, Point]]:
    """Yields `(world_points, midspans)` at each prescribed end displacement after `n_substeps`
    relaxation steps."""
    rest_motor = bodies.motor
    n_bodies = rest_motor.batch().shape[-1]
    middle = n_bodies // 2
    weightless = mv.direction([0, 0], unit=dim.length / dim.time**2)       # [] Antivector-, length/time^2
    for displacement in displacements:
        bodies = end_loaded_bodies(bodies, rest_motor, displacement)
        for _ in range(n_substeps):
            bodies = step(bodies, constraints, dt, weightless)
        midspan_motors = bodies.motor.batch()[..., middle - 1:middle + 1]  # [..., 2] Rotor+
        centres = midspan_motors >> mv.w.dual()                            # [..., 2] Antivector-
        yield world_points(bodies, shape), centres.sum(axis=-1) / 2        # [..., n_bodies] Antivector-[V], [...] Antivector-


compression_history = show_states(
    compression_states(beam_shape, beam_bodies, beam_splices, displacements, beam_dt, n_beam_substeps),
    render.beam_animation,
    beam_shape.edges,
    name="modal_xpbd_buckle",
    duration_ms=frame_duration_ms,
    n_frames=n_beam_frames,
)

Grey end girders clamp both supports while the right support advances inward, keeping the coloured interior girders straight until compressive load triggers a smooth clamped-clamped arch across the spliced joints.

In [ ]:
def critical_displacement(height: Scalar, span: Scalar) -> Scalar:
    """Euler buckling of a clamped two-chord truss of chord spacing `height` and length `span` sets
    in at end displacement `np.pi**2 * height**2 / span`."""
    return np.pi**2 * height**2 / span


midspans = stack([midspan for _, midspan in compression_history])          # [n_beam_frames, 1] Antivector-, 1 | length
critical = critical_displacement(beam_height, n_beam_girders * beam_length)
save_figure(render.buckling_figure(displacements, midspans, critical), "modal_xpbd_buckling");

The dark curve tracks midspan lateral deflection against prescribed end shortening alongside the red dotted Euler threshold, staying flat past onset while rounding perturbations grow and then rising steeply along the post-buckled branch.

In [ ]:
# checks
assert (shape.compliance > 0).all()
modal_mass = (shape.masses * shape.modes[0].dual_inverse() & shape.modes[0]).sites.sum()  # [] Scalar+
assert ((modal_mass - 1).abs() < 1e-10).all()
assert (initial_gap.dual_inverse().norm() < mv.scalar(1e-10, unit=dim.length)).all()

final_bodies = swing_history[-1][1]
final_rigid, _, _, final_gap = coupling(final_bodies, constraints)
rng = np.random.default_rng(0)
sample_forces = mv.vector(rng.normal(size=(1, constraints.ends.shape[0], 3)), unit=force_unit).cast(Force).field()  # [1] Vector+[n_constraints], force
twist_type = Twist.derive.projective(dim.length)
sample_twists = mv(twist_type, rng.normal(size=(1, n_links + 1, 3))).field()  # [1] Bivector+[n_links + 1], 1 | length
work_residual = (final_rigid.adjugate()(sample_forces) & sample_twists).sites.sum() - (sample_forces & final_rigid(sample_twists)).sites.sum()  # [1] Scalar+, energy
assert (work_residual.abs() < mv.scalar(1e-10, unit=force_unit * dim.length)).all()
assert (final_bodies.amplitudes.batch()[0, :, 1].abs() < mv.scalar(1e-12, unit=dim.length * dim.mass.root(2))).all()
assert (final_gap.dual_inverse().norm() < mv.scalar(1e-4, unit=dim.length)).all()

deflection = (mv.y & midspans[:, 0]).abs()                                 # [n_beam_frames] Scalar+, length
assert (deflection[displacements < 0.8 * critical] < mv.scalar(1e-6, unit=dim.length)).all()
assert (deflection[displacements > 2 * critical] > mv.scalar(0.5, unit=dim.length)).all()